In [1]:
# !pip install -q -U peft "transformers>=4.40" "torch>=2.0" "accelerate>=0.28"

In [2]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [5]:
from huggingface_hub import login

# Paste your Write token here
login(token="hf_YourSuperSecretTokenGoesHere") 
print("✅ Logged in successfully!")

✅ Logged in successfully!


In [ ]:
MODEL_NAME  = 'Qwen/Qwen2.5-7B-Instruct'
DPO_ADAPTER = '/kaggle/input/datasets/<<kaggle_username>>/dpo-lora-final-adapter/final_adapter'
# PUSH_NAME   = 'your-hf-username/qwen-json-extractor'  # ← change this
PUSH_NAME   = 'shield12/qwen-json-extractor'  # ← change this

# Load base in full precision for merging (needs more VRAM)
print('Loading base model...')
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
    device_map='auto', trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Load and merge DPO adapter
print('Merging DPO adapter...')
model = PeftModel.from_pretrained(base, DPO_ADAPTER)
model = model.merge_and_unload()  # this merges LoRA weights into base

Loading base model...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Merging DPO adapter...


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


In [6]:
# Push merged model to Hugging Face Hub
print(f'Pushing to Hub: {PUSH_NAME}')
model.push_to_hub(PUSH_NAME, private=False)
tokenizer.push_to_hub(PUSH_NAME, private=False)
print('✅ Model pushed! Check: https://huggingface.co/' + PUSH_NAME)

Pushing to Hub: shield12/qwen-json-extractor


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Model pushed! Check: https://huggingface.co/shield12/qwen-json-extractor
